In [12]:
import pandas as pd
import geopandas as gpd
import seaborn as sns
from matplotlib import pyplot as plt
import os
import numpy as np
from shapely.geometry import box

from land_cover.load import plot_dir, loadStolpmann21, stolpmann_indexed_pth
from land_cover.utils import create_unique_index

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
gdf = loadStolpmann21(region="na")

Save indexed file

In [13]:
gdf.to_file(stolpmann_indexed_pth)

In [3]:
gdf["loc_idx"] = create_unique_index(gdf["Latitude"], gdf["Longitude"])

In [5]:
# Globally 2167 entries
gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 1688 entries, 469 to 2166
Data columns (total 21 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   Event       65 non-null     object  
 1   SampleID    1688 non-null   object  
 2   SampleDate  1688 non-null   object  
 3   Reference   1688 non-null   object  
 4   Campaign    1688 non-null   object  
 5   Region      1688 non-null   object  
 6   Area        1688 non-null   object  
 7   Site        1688 non-null   object  
 8   Latitude    1688 non-null   float64 
 9   Longitude   1688 non-null   float64 
 10  State       1688 non-null   object  
 11  Ecoregion   1688 non-null   object  
 12  DOC         1686 non-null   float64 
 13  DeposiType  1688 non-null   object  
 14  GrIceCont   1688 non-null   object  
 15  SOC0-100    1688 non-null   float64 
 16  SOC100-200  1688 non-null   float64 
 17  SOC200-300  1688 non-null   float64 
 18  SOC0-300    1688 non-null   float64 
 19  g

In [6]:
# Note: some studies have duplicate sampleIDs at different sites
len(np.unique(gdf["SampleID"]))

1565

# How many repeat sites

In [7]:
gdf.groupby("loc_idx")["DOC"].count().unique()

array([ 1,  2,  4,  3,  5,  7,  9,  8, 12,  6])

In [ ]:
# double-check
gdf.groupby(["Latitude", "Longitude"])["DOC"].count().unique()

array([ 1,  2,  7,  3,  4,  9,  8, 12,  5,  6])

In [8]:
# select rows whose (Latitude, Longitude) occur more than once
mask = gdf.groupby("loc_idx")["DOC"].transform("count") > 1
gdf_dup_locations = gdf[mask].copy()
gdf_dup_locations

,Event,SampleID,SampleDate,Reference,Campaign,Region,Area,Site,Latitude,Longitude,...,Ecoregion,DOC,DeposiType,GrIceCont,SOC0-100,SOC100-200,SOC200-300,SOC0-300,geometry,loc_idx
611,None,Omega_Lake1989,1989,Hamilton et al. 2001,NA,Canada,Nunavut,Canadian Arctic Archipelago,81.460,-76.400,...,tundra,3.0,bedrock,high,2.1,5.8,5.6,13.5,POINT (-76.4 81.46),322
612,None,Omega_Lake1990,1990,Hamilton et al. 2001,NA,Canada,Nunavut,Canadian Arctic Archipelago,81.460,-76.400,...,tundra,2.2,glacial,high,2.1,5.8,5.6,13.5,POINT (-76.4 81.46),322
615,None,CH_Pond_A,1993,Hamilton et al. 2001,NA,Canada,Nunavut,Canadian Arctic Archipelago,72.500,-79.830,...,tundra,2.6,glacial,high,1.6,9.6,6.5,17.7,POINT (-79.83 72.5),6
616,None,CH_Pond_B,1993,Hamilton et al. 2001,NA,Canada,Nunavut,Canadian Arctic Archipelago,72.500,-79.830,...,tundra,4.2,glacial,high,1.6,9.6,6.5,17.7,POINT (-79.83 72.5),6
626,None,DV09_1994,1994,Hamilton et al. 2001,NA,Canada,Nunavut,Canadian Arctic Archipelago,75.580,-89.310,...,tundra,2.1,bedrock,high,15.0,5.8,5.6,26.4,POINT (-89.31 75.58),732
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2144,None,SS1170_2003,2003,Osburn et al. 2017,NA,Greenland,Qeqqata,Qeqqata,67.055,-51.245,...,tundra,47.0,eolian,low,10.7,9.4,6.7,26.8,POINT (-51.245 67.055),1268
2146,None,SS1250_2002,2002,Osburn et al. 2017,NA,Greenland,Qeqqata,Qeqqata,67.062,-51.190,...,tundra,16.4,eolian,low,10.7,9.4,6.7,26.8,POINT (-51.19 67.062),741
2147,None,SS1250_2003,2003,Osburn et al. 2017,NA,Greenland,Qeqqata,Qeqqata,67.062,-51.190,...,tundra,19.6,eolian,low,10.7,9.4,6.7,26.8,POINT (-51.19 67.062),741
2148,None,SS1273_2002,2002,Osburn et al. 2017,NA,Greenland,Qeqqata,Qeqqata,67.062,-51.181,...,tundra,20.2,eolian,low,10.7,9.4,6.7,26.8,POINT (-51.181 67.062),742


In [18]:
# select rows whose (Latitude, Longitude) occur more than once
# My method
grouped = gdf.groupby("loc_idx")
mask = grouped.count()["DOC"] > 1
gdf_repeat_sampling = gdf.set_index("loc_idx")[mask]
gdf_repeat_sampling

/Users/ekyzivat/mambaforge/envs/landcover/lib/python3.11/site-packages/geopandas/geodataframe.py:1750: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  result = super().__getitem__(key)


,Event,SampleID,SampleDate,Reference,Campaign,Region,Area,Site,Latitude,Longitude,State,Ecoregion,DOC,DeposiType,GrIceCont,SOC0-100,SOC100-200,SOC200-300,SOC0-300,geometry
loc_idx,,,,,,,,,,,,,,,,,,,,
322,None,Omega_Lake1989,1989,Hamilton et al. 2001,NA,Canada,Nunavut,Canadian Arctic Archipelago,81.460,-76.400,continuous,tundra,3.0,bedrock,high,2.1,5.8,5.6,13.5,POINT (-76.4 81.46)
322,None,Omega_Lake1990,1990,Hamilton et al. 2001,NA,Canada,Nunavut,Canadian Arctic Archipelago,81.460,-76.400,continuous,tundra,2.2,glacial,high,2.1,5.8,5.6,13.5,POINT (-76.4 81.46)
6,None,CH_Pond_A,1993,Hamilton et al. 2001,NA,Canada,Nunavut,Canadian Arctic Archipelago,72.500,-79.830,continuous,tundra,2.6,glacial,high,1.6,9.6,6.5,17.7,POINT (-79.83 72.5)
6,None,CH_Pond_B,1993,Hamilton et al. 2001,NA,Canada,Nunavut,Canadian Arctic Archipelago,72.500,-79.830,continuous,tundra,4.2,glacial,high,1.6,9.6,6.5,17.7,POINT (-79.83 72.5)
732,None,DV09_1994,1994,Hamilton et al. 2001,NA,Canada,Nunavut,Canadian Arctic Archipelago,75.580,-89.310,continuous,tundra,2.1,bedrock,high,15.0,5.8,5.6,26.4,POINT (-89.31 75.58)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1268,None,SS1170_2003,2003,Osburn et al. 2017,NA,Greenland,Qeqqata,Qeqqata,67.055,-51.245,continuous,tundra,47.0,eolian,low,10.7,9.4,6.7,26.8,POINT (-51.245 67.055)
741,None,SS1250_2002,2002,Osburn et al. 2017,NA,Greenland,Qeqqata,Qeqqata,67.062,-51.190,continuous,tundra,16.4,eolian,low,10.7,9.4,6.7,26.8,POINT (-51.19 67.062)
741,None,SS1250_2003,2003,Osburn et al. 2017,NA,Greenland,Qeqqata,Qeqqata,67.062,-51.190,continuous,tundra,19.6,eolian,low,10.7,9.4,6.7,26.8,POINT (-51.19 67.062)


In [15]:
gdf_repeat_sampling.to_csv(stolpmann_indexed_pth.replace(".gpkg", "_multi_sampling.csv"))

In [20]:
len(np.unique(gdf_repeat_sampling.index))

182